# AsyncGeoTIFFReader — async COG reads with `asyncio.gather`

`AsyncGeoTIFFReader` is georeader's async-native COG reader. It satisfies the
`AsyncGeoData` protocol, exposes the same metadata surface as `RasterioReader`,
and lets you fan out hundreds of reads concurrently from a single process with
`asyncio.gather`. It is a **thin (~80-LOC) adapter** over
[`developmentseed/async-geotiff`](https://github.com/developmentseed/async-geotiff):
IFD walk, tile-fetch math, decompression dispatch, and request coalescing all
live upstream — we contribute the carrier translation and protocol conformance.

**Audience.** Anyone who has used `RasterioReader` and is wondering when to
reach for the async sibling — what changes, what stays the same, what the
two-phase laziness model looks like, and how to do the things async-geotiff
deliberately doesn't (warp, resample).

This notebook runs against a small local fixture — no credentials, no network.
The patterns translate to S3 / GCS / Azure by swapping `LocalStore` for the
appropriate `obstore.store.*` class (last section shows the pseudocode).


## How `RasterioReader` fetches bytes (sync, GDAL-backed)

For context, the sync sibling routes bytes through one of three paths,
controlled by the `opener=` / `fs=` constructor knobs (see
[`bytes_path_knobs.ipynb`](bytes_path_knobs.ipynb)):

<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 760px; margin: 1em 0;">
  <div style="border: 2px solid #1976d2; border-radius: 10px; padding: 14px; background: #e3f2fd;">
    <strong>Your code</strong><br/>
    <code>reader.load()&nbsp;/&nbsp;reader.read_from_window(w)</code>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <div style="border: 2px solid #f57c00; border-radius: 10px; padding: 14px; background: #fff3e0;">
    <strong>RasterioReader</strong> &middot; wraps <code>rasterio.open(...)</code>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <table style="width: 100%; border-collapse: separate; border-spacing: 10px 0; margin: 0;">
    <tr>
      <td style="border: 2px solid #7b1fa2; border-radius: 10px; padding: 10px; background: #f3e5f5; text-align: center; vertical-align: top; width: 33%;">
        <strong>GDAL VSI</strong><br/>
        <em style="color: #555;">(default)</em><br/>
        <code>opener=None, fs=None</code><br/>
        <small style="color: #555;">libcurl in C; fastest cloud path</small>
      </td>
      <td style="border: 2px solid #7b1fa2; border-radius: 10px; padding: 10px; background: #f3e5f5; text-align: center; vertical-align: top; width: 34%;">
        <strong>Python opener</strong><br/>
        <em style="color: #555;">(custom callback)</em><br/>
        <code>opener=callable</code><br/>
        <small style="color: #555;">custom HTTP / auth, sync facade over async</small>
      </td>
      <td style="border: 2px solid #7b1fa2; border-radius: 10px; padding: 10px; background: #f3e5f5; text-align: center; vertical-align: top; width: 33%;">
        <strong>fsspec</strong><br/>
        <em style="color: #555;">(shortcut for opener)</em><br/>
        <code>fs=fsspec_fs</code><br/>
        <small style="color: #555;">niche backends, custom auth</small>
      </td>
    </tr>
  </table>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <div style="border: 2px solid #388e3c; border-radius: 10px; padding: 14px; background: #c8e6c9; text-align: center;">
    <strong>Cloud storage / local disk</strong><br/>
    <small style="color: #555;">S3, GCS, Azure, HTTPS, FTP, SFTP, GitHub, local</small>
  </div>
</div>

All three paths are **synchronous**. The reader opens a fresh `rasterio.DatasetReader`
on each `read()` call, which keeps it pickleable across `multiprocessing` /
`joblib` / Dask workers.


## How `AsyncGeoTIFFReader` fetches bytes (async, GDAL-free)

<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 760px; margin: 1em 0;">
  <div style="border: 2px solid #1976d2; border-radius: 10px; padding: 14px; background: #e3f2fd;">
    <strong>Your code</strong><br/>
    <code>await&nbsp;reader.read_from_window(w)</code>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; delegates to</div>
  <div style="border: 2px solid #f57c00; border-radius: 10px; padding: 14px; background: #fff3e0;">
    <strong>AsyncGeoTIFFReader</strong> &middot; <em style="color:#555;">this package, ~80 LOC adapter</em><br/>
    <small>Carrier translation (RasterArray &rarr; GeoTensor), protocol conformance, overview indexing</small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; wraps</div>
  <div style="border: 2px solid #c2185b; border-radius: 10px; padding: 14px; background: #fce4ec;">
    <strong>async_geotiff.GeoTIFF</strong> &middot; <em style="color:#555;">DevSeed</em><br/>
    <small>IFD walk, GeoKey parsing, GDAL-metadata parsing, tile-fetch math, RasterArray assembly</small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; (Rust core, off the event loop)</div>
  <div style="border: 2px solid #c2185b; border-radius: 10px; padding: 14px; background: #fce4ec;">
    <strong>async-tiff</strong><br/>
    <small>Per-tile range fetch &middot; decompression &middot; <strong>request coalescing for adjacent tiles</strong></small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; via</div>
  <div style="border: 2px solid #c2185b; border-radius: 10px; padding: 14px; background: #fce4ec;">
    <strong>obspec.AsyncStore</strong><br/>
    <small><code>obstore.store</code>: S3Store &middot; GCSStore &middot; AzureStore &middot; HTTPStore &middot; LocalStore</small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <div style="border: 2px solid #388e3c; border-radius: 10px; padding: 14px; background: #c8e6c9; text-align: center;">
    <strong>Cloud storage / local disk</strong>
  </div>
</div>

Boxes below `AsyncGeoTIFFReader` are all external dependencies. The Rust core
coalesces adjacent tile reads within one `await` call, so a single window read
already issues parallel HTTP range requests under the hood. The header is
fetched once on `open()`; pixel bytes are fetched per `read_*` / `load` call.


## Which reader should I use?

| Scenario | Reader | Why |
|---|---|---|
| Notebook exploration, batch scripts, single scenes | `RasterioReader` | Sync API is simpler; one read at a time is the common case. |
| JP2, NetCDF, HDF5, GRIB, non-COG formats | `RasterioReader` | Async reader is TIFF/COG only. |
| Cross-CRS reads via WarpedVRT | `RasterioReader` | Async reader does not warp (see mini-solution below). |
| `multiprocessing` / `joblib` / Dask workers | `RasterioReader` | Opens fresh per `read()`, pickleable across processes. |
| Tile servers fanning out 100s of concurrent reads | `AsyncGeoTIFFReader` | `asyncio.gather` shines when reads are network-bound. |
| Async ML inference services that read many chips per request | `AsyncGeoTIFFReader` | Concurrent fetch from one process, one event loop. |
| Cloud-heavy COG workflows that want to skip GDAL | `AsyncGeoTIFFReader` | `obstore` is Rust + HTTP/2 + native parallel ranges. |

If you're not sure, **start with `RasterioReader`**. Switch when profiling
shows you're bottlenecked on concurrent cloud reads from one process.


## Setup — build a small local COG fixture with overviews

We build a 256&times;256 tiled GeoTIFF with a 2&times;/4&times; overview ladder
so we can demonstrate both full-resolution and overview reads.

In [1]:
import os
import tempfile

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin

tmpdir = tempfile.mkdtemp()
fname = "demo.tif"
fixture_path = os.path.join(tmpdir, fname)

np.random.seed(0)
data = np.random.randint(0, 5000, size=(3, 256, 256), dtype=np.int16)

with rasterio.open(
    fixture_path, "w",
    driver="GTiff", height=256, width=256, count=3, dtype=data.dtype,
    crs="EPSG:32631", transform=from_origin(500000.0, 4600000.0, 10.0, 10.0),
    tiled=True, blockxsize=64, blockysize=64, compress="deflate",
    nodata=0,
) as dst:
    dst.write(data)

# Build a 2x / 4x overview ladder so we can read at three resolutions.
with rasterio.open(fixture_path, "r+") as ds:
    ds.build_overviews([2, 4], Resampling.average)
    ds.update_tags(ns="rio_overview", resampling="average")

print(f"Fixture: {fixture_path}")


Fixture: /var/folders/k9/_v6ywhvj0nq36tpttd3j4mq80000gn/T/tmp9ka3ou11/demo.tif


## Opening a reader — two-phase laziness

Construction is **two phases**:

1. **`AsyncGeoTIFFReader(path, store=...)`** — pure constructor, no I/O.
   Property accessors raise `RuntimeError` before `open()`.
2. **`await AsyncGeoTIFFReader.open(...)`** — fetches only the COG header
   (the IFD chain). Cheap. After this, sync metadata properties are instant.

Pixel bytes are fetched on the *first* `await reader.read_*(...)` /
`await reader.load()` call.


In [2]:
from obstore.store import LocalStore
from georeader.async_geotiff_reader import AsyncGeoTIFFReader

store = LocalStore(prefix=tmpdir)

# Phase 1: construct (no I/O). Properties raise until open() is awaited.
reader = AsyncGeoTIFFReader(fname, store=store)
print(f"Before open: {reader}")

try:
    _ = reader.crs
except RuntimeError as e:
    print(f"  reader.crs raised: {e}")


Before open: AsyncGeoTIFFReader(path_or_url='demo.tif', overview_level=None, unopened)
  reader.crs raised: AsyncGeoTIFFReader not opened — call `await AsyncGeoTIFFReader.open(...)` or use `async with`.


In [3]:
# Phase 2: open() fetches the IFD chain only. Pixels are NOT yet downloaded.
reader = await AsyncGeoTIFFReader.open(fname, store=store)
print(f"After open:  {reader}")


After open:  AsyncGeoTIFFReader(path_or_url='demo.tif', overview_level=None, opened)


## Sync metadata properties (after `open()`)

Same surface as `RasterioReader`: `crs`, `transform`, `shape`, `width`,
`height`, `bounds`, `res`, `dtype`, `fill_value_default`, `dims`, plus the
`footprint(crs)` method. All sync, all instant — they just read fields off
the already-fetched header.

In [4]:
print(f"crs       : {reader.crs}")
print(f"transform : {reader.transform}")
print(f"shape     : {reader.shape}    (count, height, width)")
print(f"dtype     : {reader.dtype}")
print(f"bounds    : {reader.bounds}")
print(f"res       : {reader.res}")
print(f"dims      : {reader.dims}")
print(f"fill_value_default: {reader.fill_value_default}")


crs       : EPSG:32631
transform : | 10.00, 0.00, 500000.00|
| 0.00,-10.00, 4600000.00|
| 0.00, 0.00, 1.00|
shape     : (3, 256, 256)    (count, height, width)
dtype     : int16
bounds    : (500000.0, 4597440.0, 502560.0, 4600000.0)
res       : (10.0, 10.0)
dims      : ['band', 'y', 'x']
fill_value_default: 0.0


## Reading data

Three async read methods, each returning a `GeoTensor` (numpy-subclass
carrier with `.values`, `.transform`, `.crs`, ...):

| Method | What it reads |
|---|---|
| `await reader.load()` | The whole raster at the current `overview_level`. |
| `await reader.read_from_window(window)` | A `rasterio.windows.Window` region. |
| `await reader.read_from_bounds(bounds)` | A geographic-bounds region, *native CRS only*. |

The result is numerically identical to `RasterioReader.read_from_window` on
the same window — let's verify:

In [5]:
import rasterio.windows
from georeader.rasterio_reader import RasterioReader

win = rasterio.windows.Window(col_off=32, row_off=16, width=64, height=48)

async_gt = await reader.read_from_window(win)
sync_gt = RasterioReader(fixture_path).read_from_window(win).load()

print(f"async_gt.values.shape: {async_gt.values.shape}")
print(f"sync_gt.values.shape:  {sync_gt.values.shape}")
print(f"Numerically identical: {np.array_equal(async_gt.values, sync_gt.values)}")


async_gt.values.shape: (3, 48, 64)
sync_gt.values.shape:  (3, 48, 64)
Numerically identical: True


## Overviews — reading at lower resolutions

COGs commonly ship with a pyramid of pre-downsampled overviews. Each overview
is a smaller copy of the full raster, useful for quick previews, tile-server
rendering at low zoom, or saving bandwidth when you don't need full detail.

`AsyncGeoTIFFReader` exposes overviews via the `overview_level` constructor
kwarg:

- `overview_level=None` (default) &mdash; read from the **primary IFD**
  (full resolution).
- `overview_level=i` &mdash; read from the **i-th overview** (0-based). For
  a `[2, 4]` ladder, `overview_level=0` is the 2&times;-decimated layer and
  `overview_level=1` is the 4&times;-decimated one.

Properties (`shape`, `transform`, `res`, ...) reflect the active level.
Reads happen against the corresponding pixel grid &mdash; you don't pay for
the bytes of higher-resolution levels.

In [6]:
# Inspect what overviews this COG has
n_overviews = len(reader._geotiff.overviews)
print(f"This COG has {n_overviews} overview(s)")

# Open readers at three resolutions: full-res, 2x-decimated, 4x-decimated
reader_full = await AsyncGeoTIFFReader.open(fname, store=store)               # primary IFD
reader_ovr0 = await AsyncGeoTIFFReader.open(fname, store=store, overview_level=0)
reader_ovr1 = await AsyncGeoTIFFReader.open(fname, store=store, overview_level=1)

print()
print(f"Full resolution  : shape={reader_full.shape}, res={reader_full.res}")
print(f"Overview 0 (2x)  : shape={reader_ovr0.shape}, res={reader_ovr0.res}")
print(f"Overview 1 (4x)  : shape={reader_ovr1.shape}, res={reader_ovr1.res}")


This COG has 2 overview(s)

Full resolution  : shape=(3, 256, 256), res=(10.0, 10.0)
Overview 0 (2x)  : shape=(3, 128, 128), res=(20.0, 20.0)
Overview 1 (4x)  : shape=(3, 64, 64), res=(40.0, 40.0)


In [7]:
# Loading the whole raster at each level pays bytes proportional to grid size
gt_full = await reader_full.load()
gt_ovr0 = await reader_ovr0.load()
gt_ovr1 = await reader_ovr1.load()

print(f"Full-res load: shape={gt_full.values.shape}, bytes={gt_full.values.nbytes:>8,}")
print(f"Overview 0   : shape={gt_ovr0.values.shape}, bytes={gt_ovr0.values.nbytes:>8,}  ({gt_full.values.nbytes/gt_ovr0.values.nbytes:.1f}x smaller)")
print(f"Overview 1   : shape={gt_ovr1.values.shape}, bytes={gt_ovr1.values.nbytes:>8,}  ({gt_full.values.nbytes/gt_ovr1.values.nbytes:.1f}x smaller)")


Full-res load: shape=(3, 256, 256), bytes= 393,216
Overview 0   : shape=(3, 128, 128), bytes=  98,304  (4.0x smaller)
Overview 1   : shape=(3, 64, 64), bytes=  24,576  (16.0x smaller)


## Concurrent fan-out — the killer feature

The point of going async is to fan out many reads concurrently from one
process. `asyncio.gather` does it in one line.

**Honest disclaimer about local fixtures:** speedups from async only show up
against meaningful per-read latency &mdash; typically *cloud* reads. Against
a local file, the overhead can even dominate. Don't draw timing conclusions
from this cell; it proves the fan-out *works*, which is the actual question.
Real wins arrive when each `read_from_window` is a 50&ndash;200 ms network
round-trip and you have 100+ of them to issue.


In [8]:
import asyncio

# 16 non-overlapping 64x64 windows tiling the 256x256 raster
windows = [
    rasterio.windows.Window(col_off=c, row_off=r, width=64, height=64)
    for r in range(0, 256, 64) for c in range(0, 256, 64)
]

results = await asyncio.gather(*[reader.read_from_window(w) for w in windows])

print(f"Issued {len(windows)} concurrent reads against one reader")
print(f"All shapes correct: {all(r.values.shape == (3, 64, 64) for r in results)}")


Issued 16 concurrent reads against one reader
All shapes correct: True


## `async with` — context manager

When you don't want to manage `open()` / `aclose()` yourself, use the async
context manager. It opens lazily on enter and runs `aclose()` on exit
(currently a no-op since `obstore` pools its own connections).

In [9]:
ctx_reader = AsyncGeoTIFFReader(fname, store=store)

async with ctx_reader:
    gt = await ctx_reader.load()
    print(f"Inside the context: shape={gt.values.shape}, dtype={gt.values.dtype}")


Inside the context: shape=(3, 256, 256), dtype=int16


## What this reader does NOT do

`async-geotiff` explicitly disclaims warp / resample / overview
auto-selection. We follow the same boundary &mdash; calling
`read_from_bounds(target_crs=...)` or `read_from_bounds(target_resolution=...)`
raises `NotImplementedError` with a clear error message:

In [10]:
try:
    await reader.read_from_bounds(reader.bounds, target_crs="EPSG:4326")
except NotImplementedError as e:
    print(f"NotImplementedError: {e}")


NotImplementedError: AsyncGeoTIFFReader does not warp or resample. Read in the native CRS, then call georeader.read.read_reproject_like, or use RasterioReader for WarpedVRT-based on-the-fly warping.


### Mini-solution: warp / reproject **after** loading

When you need a different CRS or resolution, the recommended pattern is
**fetch native, then warp post-step** via georeader's sync warp helpers. Two
shapes cover most cases:

- **`read.read_to_crs(gt, dst_crs=...)`** &mdash; reproject a loaded
  `GeoTensor` to a target CRS with a derived transform.
- **`read.read_reproject_like(gt, gt_target)`** &mdash; reproject onto the
  exact grid of another `GeoTensor` (matching extent, resolution, CRS).

Both are sync and use `rasterio.warp` under the hood (which means they pull
GDAL into the dependency cone &mdash; that is the cost of warping).

In [11]:
# 1. Read native CRS via the async reader
gt_native = await reader.load()
print(f"Native: crs={gt_native.crs}, shape={gt_native.values.shape}, transform={gt_native.transform}")

# 2a. Warp to a target CRS (sync post-step)
from georeader import read

gt_wgs84 = read.read_to_crs(gt_native, dst_crs="EPSG:4326")
print(f"Warped to WGS84: crs={gt_wgs84.crs}, shape={gt_wgs84.values.shape}")


Native: crs=EPSG:32631, shape=(3, 256, 256), transform=| 10.00, 0.00, 500000.00|
| 0.00,-10.00, 4600000.00|
| 0.00, 0.00, 1.00|


Warped to WGS84: crs=EPSG:4326, shape=(3, 218, 290)


In [12]:
# 2b. Reproject onto another GeoTensor's grid (e.g. for stack alignment)
from georeader.geotensor import GeoTensor

target_grid = GeoTensor(
    values=np.zeros((3, 200, 200), dtype=np.int16),
    transform=from_origin(500000.0, 4600000.0, 12.0, 12.0),  # 12m pixels instead of 10m
    crs="EPSG:32631",
)

gt_aligned = read.read_reproject_like(gt_native, target_grid)
print(f"Aligned to target grid: shape={gt_aligned.values.shape}, transform={gt_aligned.transform}")


Aligned to target grid: shape=(3, 200, 200), transform=| 12.00, 0.00, 500000.00|
| 0.00,-12.00, 4600000.00|
| 0.00, 0.00, 1.00|


## Tips and gotchas

- **Two-phase laziness.** Header on `open()`, pixels on `read()`. Properties
  raise `RuntimeError` before `open()`. Use `async with` if you want to skip
  the explicit `open` + `aclose` dance.
- **Not pickleable across processes.** The `_geotiff` handle is alive
  between reads (faster repeated reads) but won't survive a
  `multiprocessing` / `joblib` / Dask worker boundary. For multi-process,
  open the reader fresh in each worker, or use `RasterioReader`.
- **`store=` is required &mdash; no default.** Pick the right `obstore` Store per
  cloud:
  - `obstore.store.S3Store(bucket=..., region=...)` for AWS S3
  - `obstore.store.GCSStore(bucket=..., ...)` for Google Cloud Storage
  - `obstore.store.AzureStore(container_name=..., ...)` for Azure Blob
  - `obstore.store.LocalStore(prefix=dir)` for local disk
  - `obstore.store.HTTPStore.from_url(url)` for HTTPS
- **Overviews.** `overview_level=None` (default) reads full resolution;
  `overview_level=i` reads the i-th overview (0-based). The COG must
  actually have overviews — `overview_level=0` on a non-overview file
  raises `IndexError`. Auto-picking the right level for a target resolution
  isn't done for you; pick explicitly based on `len(reader._geotiff.overviews)`.
- **TIFF/COG only.** For JP2, NetCDF, HDF5, GRIB, use `RasterioReader`.
- **Mask convention.** `async-geotiff`'s `RasterArray.mask` uses
  `True = valid` (inverse of numpy MA's convention). The adapter handles
  this and substitutes `fill_value_default` where invalid.
- **No warp / resample / overview auto-selection.** Use the mini-solution
  above (load native + `read.read_to_crs` / `read.read_reproject_like`), or
  reach for `RasterioReader` with WarpedVRT for one-shot on-the-fly warping.


## Going to the cloud (pseudocode)

The flow is identical &mdash; swap `LocalStore` for the appropriate
`obstore.store.*` class. The reader doesn't care which cloud is behind the
store.

```python
from obstore.store import S3Store
from georeader.async_geotiff_reader import AsyncGeoTIFFReader

store = S3Store(bucket="my-bucket", region="us-east-1")
reader = await AsyncGeoTIFFReader.open("scene.tif", store=store)

# All the same methods work
gt = await reader.load()

# Fan out across many windows from one bucket:
chips = await asyncio.gather(*[reader.read_from_window(w) for w in windows])
```

For credentials, look at the relevant `obstore.store.*` constructor — each
store accepts the standard cloud-specific auth: env-vars, IAM roles, SAS
tokens, etc. See the
[obstore docs](https://developmentseed.org/obstore/latest/) for the full
matrix.


## Cleanup

In [13]:
import shutil
shutil.rmtree(tmpdir)
